In [ ]:
import pandas as pd
import random

In [ ]:
zones = pd.read_csv("zones.csv")
zones = zones.set_index("PID")
zones.shape

In [ ]:
zones.head()

In [ ]:
preferences = pd.read_csv("preferences.csv")
preferences = preferences.set_index("Your PID")
preferences.shape

In [ ]:
preferences.head()

In [ ]:
merged = zones.merge(preferences, left_index=True, right_on="Your PID", how="left")
merged

In [ ]:
import math

def listify_preferences(value):
    preferences = []
    ordinals = ['First']# , 'Second', 'Third', 'Fourth'] #, 'Fifth']
    for ordinal in ordinals:
        preference = value[f"(Optional) {ordinal} Choice Partner's PID"]
        if not math.isnan(preference):
            preferences.append(preference)
    value["preferences"] = preferences
    return value


data = merged.to_dict(orient='records')
data = [listify_preferences(value) for value in data]
data

In [ ]:
people = {d['Your PID']: d for d in data}
people

In [ ]:
TEAM_SIZE = 2

teams = []
assigned = ()
for _, person in people.items():
    person['team'] = None # Reset Team each run
    person['choices'] = [int(choice) for choice in person['preferences'][:]] # Copy preferences each run

# Goal: Take a pass and identify mutual top matches of remaining preferences. Pop those preferences and drop into a team.
def match_top_pass(people) -> int:
    top_choices = {pid: int(person['choices'][0]) for pid, person in people.items() if len(person['choices']) > 0}
    mutual_top = [(pid, partner) for pid, partner in top_choices.items() if partner in top_choices and top_choices[partner] == pid]
    for pid_a, pid_b in mutual_top:
        person_a = people[pid_a]
        person_b = people[pid_b]
        if person_a['team'] is None and person_b['team'] is None:
            new_team = {pid_a: person_a, pid_b: person_b}
            person_a['team'] = person_b['team'] = new_team
            teams.append(new_team)
        else:
            if person_a['team'] and person_b['team'] is None:
                team = person_a['team']
                if pid_b not in team and len(team) < TEAM_SIZE:
                    team[pid_b] = person_b
                    person_b['team'] = team
            elif person_b['team'] and person_a['team'] is None:
                team = person_b['team']
                if pid_a not in team and len(team) < TEAM_SIZE:
                    team[pid_a] = person_a
                    person_a['team'] = team
        # Pop Top Prefence:
        person_a['choices'] = person_a['choices'][1:]
        person_b['choices'] = person_b['choices'][1:]
    return len(mutual_top)

# Goal - Match anyone else in list without a match wherever possible
def mutual_where_possible(people) -> int:
    choices = {pid: person['choices'] for pid, person in people.items() if len(person['choices']) > 0}
    for pid_a, choices in choices.items():
        for pid_b in choices:
            if pid_b in people and people[pid_b]['team'] is None:
                person_b = people[pid_b]
                if pid_a in person_b['choices']:
                    person_a = people[pid_a]
                    if person_a['team'] is None and person_b['team'] is None:
                        new_team = {pid_a: person_a, pid_b: person_b}
                        person_a['team'] = person_b['team'] = new_team
                        teams.append(new_team)
                    else:
                        if person_a['team'] and person_b['team'] is None:
                            team = person_a['team']
                            if pid_b not in team and len(team) < TEAM_SIZE:
                                team[pid_b] = person_b
                                person_b['team'] = team
                        elif person_b['team'] and person_a['team'] is None:
                            team = person_b['team']
                            if pid_a not in team and len(team) < TEAM_SIZE:
                                team[pid_a] = person_a
                                person_a['team'] = team

mutual_where_possible(people)



from collections import Counter

# Let's try and bring together mutual pairs of 2
def reduce_teams(teams, limit=1) -> list[dict[int, dict]]:
    reduced = []
    i = 0
    for i in range(0, len(teams)):
        team = teams[i]
        choices2d = [person['choices'] for _, person in team.items()]
        choices = [choice for choices in choices2d for choice in choices]
        pids = [pid for pid, count in Counter(choices).items() if count > limit]
        if len(pids) > 0:
            for pid in pids:
                person = people[pid]
                if person['team'] and person['team'] != team and len(team) + len(person['team']) <= TEAM_SIZE:
                    # merge teams
                    other_team = person['team']
                    for pid, person in other_team.items():
                        team[pid] = person
                        person['team'] = team
                    other_team.clear()
        if len(team) > 0:
            reduced.append(team)
    return reduced


while match_top_pass(people) > 0:
    ...

teams = list(filter(len, reduce_teams(teams)))
teams = list(filter(len, reduce_teams(teams, 0)))

# Set soloists without teams, but who submitted preferences, up as single teams ahead of next pass
for pid, person in people.items():
    if person['team'] is None and len(person['choices']) > 0:
        team = {pid: person}
        teams.append(team)
        person['team'] = team

# Goal: Find matches with students who didn't submit preferences
def match_free_agents(people) -> int:
    matched = 0
    for pid, person in people.items():
        team = person['team']
        if team != None and len(team) < TEAM_SIZE:
            choices = person['choices']
            for choice_pid in choices:
                if choice_pid in people and people[choice_pid]['team'] is None and len(team) < TEAM_SIZE:
                    people[choice_pid]['team'] = team
                    team[choice_pid] = people[choice_pid]
                    matched += 1
    return matched

while match_free_agents(people) > 0:
    ...

# Find soloists
free_agents = []
next_team = {}
for pid, person in people.items():
    if person['team'] is None:
        if len(next_team) == TEAM_SIZE:
            print(next_team)
            teams.append(next_team)
            next_team = {}
        next_team[pid] = person
        person['team'] = next_team

if len(next_team) > 0:
    teams.append(next_team)

teams = list(sorted(teams, key=lambda team: len(team), reverse=True))
print(f"Teams: {len(teams)}")
print([len(team) for team in teams])
print(f"People on a team: {sum([len(team) for team in teams])}")
print(f"People in the course: {len(people)}")

import csv
with open('teams.csv', 'w') as file:
    writer = csv.writer(file)
    for team in teams:
        row = [len(team)]
        for pid, student in team.items():
            row.append(pid)
            row.append(student['Name'])
            row.append(student['Zone'])
        writer.writerow(row)


In [ ]:

raise KeyError("ORIGINAL")
# Let's find mutual matches...

pairs = []
paired = set()
for key, value in people.items():
    try:
        preference_pid = int(value['partner'])
        if preference_pid in people and int(people[preference_pid]['partner']) == key:
            if key not in paired and preference_pid not in paired and key != preference_pid:
                pairs.append((key, preference_pid))
                paired.add(key)
                paired.add(preference_pid)
    except:
        ...

print(f'{len(pairs) * 2} two-way paired')
remaining = {k: v for k, v in people.items() if k not in paired}
print(f"{len(remaining)} remaining")

# Now lets find one-way matches
keys = list(remaining.keys())
random.shuffle(keys)
remaining = {key: remaining[key] for key in keys}
for key, value in remaining.items():
    if key in paired: continue
    try:
        preference_pid = int(value['partner'])
        if preference_pid in remaining and preference_pid not in paired and value['Zone'] == remaining[preference_pid]['Zone']:
            if key != preference_pid:
                pairs.append((key, preference_pid))
                paired.add(key)
                paired.add(preference_pid)
    except Exception as e:
        ...

print(f'{len(pairs) * 2} one-way paired')
remaining = {k: v for k, v in remaining.items() if k not in paired}
print(f"{len(remaining)} remaining")

# Next, let's find matches within same zone

for key, value in remaining.items():
    if key not in paired:
        zone = value["Zone"]
        for inner_key, inner_value in remaining.items():
            if key != inner_key and inner_key not in paired:
                if zone == inner_value["Zone"]:
                    pairs.append((key, inner_key))
                    paired.add(key)
                    paired.add(inner_key)
                    break

print(f'{len(pairs) * 2} zone paired')
remaining = {k: v for k, v in remaining.items() if k not in paired}
print(f"{len(remaining)} remaining")

remaining_values = list(remaining.values())
remaining_values.sort(key=lambda p: p['Zone'])
remaining_list = list(map(lambda p: p['Your PID'], remaining_values))
print(remaining_list)
for i in range(0, len(remaining_list) - 1, 2):
    a = remaining_list[i]
    b = remaining_list[i + 1]
    pairs.append((a, b))
    paired.add(a)
    paired.add(b)

print(f'{len(pairs) * 2} zone paired')
remaining = {k: v for k, v in remaining.items() if k not in paired}
print(f"{len(remaining)} remaining")
print("Remaining:")
print(remaining)

print("====")

# Setup the rows for a CSV
output = []
for pair in pairs:
    a = people[pair[0]]
    b = people[pair[1]]
    output.append({'(A) Name': a['Name'], '(B) Name': b['Name']})

print(output)

In [ ]:
df = pd.DataFrame(output)
df.to_csv("pairs.csv")
df.head()